<a href="https://colab.research.google.com/github/vardhan23v/agentic-ai/blob/main/03_multi_agent_crewai_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  AI Topic Summarizer using CrewAI (Multi-Agent Workflow)

Welcome to this project, where we demonstrate the power of **multi-agent collaboration** using [CrewAI](https://www.crewai.com/). In this notebook, you'll see how a team of AI agents — powered by a Language Model (LLM) — can work together to research and summarize any AI topic in a structured, exam-friendly format.

## Project Objective

To build a smart and modular system that:
- Accepts an **AI topic** from the user
- Uses one agent to **research** the topic
- Uses another agent to **summarize** the research into clean, indexed points
- Optionally uses a third agent to **validate** or **refine** the output

Why Multi-Agent?

Instead of using one AI for everything, we split the workload:
- Each agent specializes in a specific task
- Tasks are clearer and outputs are more structured
- Mimics real-world teamwork — where one person researches, another writes, another edits

##  Install CrewAI Library

We start by installing the `crewai` library along with optional tools.  
This library helps us build collaborative agents powered by Language Models.

>  This step only needs to be run once (and may already be pre-installed in some environments).


In [25]:
%pip install -q crewai[tools]
%pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.6 MB/s eta 0:00:00


##  Import Required Modules

We import the core classes from **CrewAI**:

- `LLM`: The language model to be used by agents
- `Agent`: Defines individual AI agents
- `Task`: Describes what each agent must do
- `Crew`: Manages the coordination between agents

We also import Python's built-in `textwrap` module to help format long text for better readability.


In [26]:
from crewai import LLM, Agent, Task, Crew
import textwrap

##  Set Up the Language Model

We configure the LLM (`gpt-4.1-nano`) using Nexus API.  
This model will power all our agents.


In [27]:
llm = LLM(
    model="openai/nova-micro",
    temperature=0.7,
    base_url="https://nexusapi.navigatelabs.ai",
    api_key="")

##  Define the AI Agents

We create three agents, each with a specific role:
- `research_agent`: Gathers key facts about the topic
- `summarizer_agent`: Writes a clear, structured summary
- `tester_agent`: Reviews and improves the summary


In [28]:
resume_agent = Agent(
    role="Resume Analysis Expert",
    goal="Analyze the resume and extract important information including skills, education, experience, projects, and achievements",
    backstory="An expert resume analyst who carefully examines resumes and identifies the most important candidate information.",
    tools=[],
    llm=llm,
    verbose=True
)
skills_agent = Agent(
    role="Skills and Career Analyst",
    goal="Analyze the candidate's skills and determine how well they match the target job role",
    backstory="An experienced technical recruiter who evaluates technical skills, projects, experience, and career alignment.",
    tools=[],
    llm=llm,
    verbose=True
)
reviewer_agent = Agent(
    role="Resume Reviewer",
    goal="Review the resume analysis and provide clear suggestions to improve the resume",
    backstory="An expert resume reviewer who checks resumes for clarity, structure, ATS compatibility, and professional presentation.",
    tools=[],
    llm=llm,
    verbose=True
)

## Get User Input

Ask the user to enter the AI topic to be summarized.


In [29]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print(f"✅ Resume uploaded: {pdf_path}")
job_role = input(
    "Enter the job role you are applying for: "
)

Saving resume.pdf to resume (2).pdf
✅ Resume uploaded: resume (2).pdf
Enter the job role you are applying for: full stack developer


##  Define the Tasks for Each Agent

- `research_task`: Collects detailed info about the topic  
- `summary_task`: Converts research into a clean, indexed summary  
- `testing_task`: Reviews and enhances the summary for clarity and completeness

In [32]:
from pypdf import PdfReader

# Extract text from the uploaded PDF
reader = PdfReader(pdf_path)
resume_text = ""
for page in reader.pages:
    resume_text += page.extract_text() + "\n"

resume_task = Task(
    description=f"""
Analyze the following resume for the target job role: '{job_role}'.

Resume:
{resume_text}

Extract and analyze:

1. Education
2. Technical skills
3. Programming languages
4. Frameworks and libraries
5. Tools and technologies
6. Work experience
7. Projects
8. Certifications
9. Achievements
10. Strengths

Do not invent any information that is not present in the resume.
""",

    expected_output="""
A structured resume analysis covering education, technical skills,
programming languages, frameworks, tools, experience, projects,
certifications, achievements, and strengths.
""",

    agent=resume_agent
)


skills_task = Task(
    description=f"""
Using the resume analysis, evaluate the candidate's profile
for the target job role: '{job_role}'.

Identify:

1. Relevant technical skills
2. Relevant projects
3. Relevant experience
4. Skills that match the job role
5. Important skills that are missing
6. Weak areas in the resume
7. Technologies that could be learned
8. Career improvement suggestions

Base the analysis only on the information available in the resume.
Do not invent skills or experience.
""",

    expected_output="""
A structured job-match analysis containing relevant skills,
matching skills, missing skills, weak areas, and practical
improvement suggestions.
""",

    agent=skills_agent,
    context=[resume_task]
)


review_task = Task(
    description=f"""
Review the resume analysis and skills analysis for the
target job role: '{job_role}'.

Create a final resume improvement report containing:

1. Resume strengths
2. Resume weaknesses
3. ATS optimization suggestions
4. Important keywords for the target role
5. Project improvement suggestions
6. Experience improvement suggestions
7. Formatting suggestions
8. Skills that should be highlighted
9. Final improvement checklist

Do not invent qualifications, experience, projects, or skills.
""",

    expected_output="""
A final polished resume review containing strengths, weaknesses,
ATS optimization suggestions, relevant keywords, and actionable
improvements.
""",

    agent=reviewer_agent,
    context=[resume_task, skills_task]
)

##  Run the Crew

The agents now work together to complete the tasks and generate the final summary.


In [33]:
crew = Crew(
    agents=[
        resume_agent,
        skills_agent,
        reviewer_agent
    ],
    tasks=[
        resume_task,
        skills_task,
        review_task
    ],
    verbose=True
)

result = await crew.kickoff_async()

print("\n✅ Final Resume Analysis:\n")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1b03a553-9ac2-4a22-968e-bf73f952ee4e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Analyze the following resume for the target job role: 'full stack developer'.                                  │
│                                                                                                                 │
│  Resume:                                                                                                        │
│  VALLUR SREE VARDHAN                                                                                            │
│  +91 98859 23035|23vvardhan@gmail.com|linkedin.com/in/vardhan-v23|github.com/vardhan23v|                        │
│  Kurnool, Andhra Pradesh, India                                                                                 │
│  Professional Summary                                                                                           │
│  Computer Science student who builds and ships full-stack web apps powered by AI. Built a Chrome extension      │
│  generator using multiple AI models (Claude, Gemini, Groq) and an AI-based code review tool using Claude.       │
│  Skilled in JavaScript, Python, React.js, Node.js, and SQL, with real, hands-on experience connecting AI APIs   │
│  into                                                                                                           │
│  live, working products.                                                                                        │
│  Experience                                                                                                     │
│  AI Product Beta Tester – Oxlo.aiJul 2026 – Present                                                             │
│  Part-time                                                                                                      │
│  • Selected for the OxCode Founding Builders community to test AI software engineering tools ahead of public    │
│  release.                                                                                                       │
│  • Evaluated product functionality and AI-assisted developer workflows, reporting bugs to inform pre-launch     │
│  iter-                                                                                                          │
│  ation.                                                                                                         │
│  Education                                                                                                      │
│  N M A M Institute of Technology (NITTE), Karnataka2024 – 2028                                                  │
│  B.Tech – Computer Science and Engineering                                                                      │
│  Narayana Junior College, Kurnool2022 – 2024                                                                    │
│  Class XII (10+2) – MPC                                                                                         │
│  Narayana English Medium School, Kurnool2022                                                                    │
│  Class X – SSC                                                                                                  │
│  Technical Skills                                                                                               │
│  Languages:JavaScript, TypeScript (working knowledge), Python, HTML5, CSS3, SQL, C                              │
│  Frontend:React.js, Vite, Responsive Design, DOM Manipulation                                                   │
│  Backend:Node.js, Express.js, RESTful APIs, JWT Authent

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resume Analysis Expert                                                                                  │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Analyze the following resume for the target job role: 'full stack developer'.                                  │
│                                                                                                                 │
│  Resume:                                                                                                        │
│  VALLUR SREE VARDHAN                                                                                            │
│  +91 98859 23035|23vvardhan@gmail.com|linkedin.com/in/vardhan-v23|github.com/vardhan23v|                        │
│  Kurnool, Andhra Pradesh, India                                                                                 │
│  Professional Summary                                                                                           │
│  Computer Science student who builds and ships full-stack web apps powered by AI. Built a Chrome extension      │
│  generator using multiple AI models (Claude, Gemini, Groq) and an AI-based code review tool using Claude.       │
│  Skilled in JavaScript, Python, React.js, Node.js, and SQL, with real, hands-on experience connecting AI APIs   │
│  into                                                                                                           │
│  live, working products.                                                                                        │
│  Experience                                                                                                     │
│  AI Product Beta Tester – Oxlo.aiJul 2026 – Present                                                             │
│  Part-time                                                                                                      │
│  • Selected for the OxCode Founding Builders community to test AI software engineering tools ahead of public    │
│  release.                                                                                                       │
│  • Evaluated product functionality and AI-assisted developer workflows, reporting bugs to inform pre-launch     │
│  iter-                                                                                                          │
│  ation.                                                                                                         │
│  Education                                                                                                      │
│  N M A M Institute of Technology (NITTE), Karnataka2024 – 2028                                                  │
│  B.Tech – Computer Science and Engineering                                                                      │
│  Narayana Junior College, Kurnool2022 – 2024                                                                    │
│  Class XII (10+2) – MPC                                                                                         │
│  Narayana English Medium School, Kurnool2022                                                                    │
│  Class X – SSC                                                                                                  │
│  Technical Skills                                                                                               │
│  Languages:JavaScript, TypeScript (working knowledge), Python, HTML5, CSS3, SQL, C                              │
│  Frontend:React.js, Vite, Responsive Design, DOM Manipu

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resume Analysis Expert                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Resume Analysis for Vallur Sree Vardhan                                                                    │
│                                                                                                                 │
│  #### Education                                                                                                 │
│  - **N M A M Institute of Technology (NITTE), Karnataka**                                                       │
│    - B.Tech – Computer Science and Engineering                                                                  │
│    - Expected Graduation: 2028                                                                                  │
│  - **Narayana Junior College, Kurnool**                                                                         │
│    - Class XII (10+2) – MPC                                                                                     │
│    - 2022 – 2024                                                                                                │
│  - **Narayana English Medium School, Kurnool**                                                                  │
│    - Class X – SSC                                                                                              │
│    - 2022                                                                                                       │
│                                                                                                                 │
│  #### Technical Skills                                                                                          │
│  - Languages: JavaScript, TypeScript (working knowledge), Python, HTML5, CSS3, SQL, C                           │
│  - Frontend: React.js, Vite, Responsive Design, DOM Manipulation                                                │
│  - Backend: Node.js, Express.js, RESTful APIs, JWT Authentication                                               │
│  - Databases: MongoDB, MySQL, Schema Design, CRUD Operations                                                    │
│  - AI/APIs: Claude API, Gemini API, Groq API, Prompt Engineering, LLM Integration                               │
│  - Tools: Git, GitHub, Vercel, VS Code, Postman                                                                 │
│                                                                                                                 │
│  #### Programming Languages                                                                                     │
│  - JavaScript, TypeScript, Python, HTML5, CSS3, SQL, C                                                          │
│                                                                                                                 │
│  #### Frameworks and Libraries                                                                                  │
│  - Frontend: React.js                                                                                           │
│  - Backend: Node.js, Express.js                                                                                 │
│                                                                                                                 │
│  #### Tools and Technologies                                                                                    │
│  - **Development Tools:** Git, GitHub, Vercel, VS Code,

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resume Reviewer                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Resume Improvement Report for Vallur Sree Vardhan                                                          │
│                                                                                                                 │
│  #### 1. Resume Strengths                                                                                       │
│                                                                                                                 │
│  - **Technical Proficiency**: Vallur possesses a solid foundation in both frontend and backend technologies,    │
│  with strong skills in React.js, Node.js, Express.js, and database management with MongoDB and MySQL.           │
│  - **Project Management**: Demonstrated project management skills through leading the development of            │
│  AI-powered code review tools and a Chrome extension generator.                                                 │
│  - **AI Integration**: Extensive experience with integrating AI-related APIs and practical application in       │
│  real-world projects.                                                                                           │
│  - **Certifications**: Holds relevant certifications that enhance technical and practical skills, such as       │
│  those from Forage and Infosys Springboard.                                                                     │
│  - **Active Participation**: Active participant in beta testing communities, contributing to the development    │
│  of innovative tools.                                                                                           │
│                                                                                                                 │
│  #### 2. Resume Weaknesses                                                                                      │
│                                                                                                                 │
│  - **Lack of Professional Work Experience**: No full-time professional work experience apart from the beta      │
│  testing role, which might be a concern for some employers.                                                     │
│  - **Advanced Topics**: Limited exposure to advanced topics such as machine learning, database optimization,    │
│  and large-scale system architecture.                                                                           │
│  - **DevOps and Cloud Experience**: No mention of experience with CI/CD pipelines, containerization,            │
│  orchestration, or major cloud platforms like AWS, GCP, or Azure.                                               │
│  - **Security Skills**: No specific skills mentioned in advanced security practices or compliance.              │
│                                                                                                                 │
│  #### 3. ATS Optimization Suggestions                                                                           │
│                                                                                                                 │
│  - **Keywords**: Use keywords directly from the job description in the resume. Incorporate terms like "full     │
│  stack developer," "React.js," "Node.js," "Express.js," "RESTful APIs," "JWT Authentication," "MongoDB,"        │
│  "MySQL," "CI/CD," "Docker," "AWS," "security practices

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Review the resume analysis and skills analysis for the                                                         │
│  target job role: 'full stack developer'.                                                                       │
│                                                                                                                 │
│  Create a final resume improvement report containing:                                                           │
│                                                                                                                 │
│  1. Resume strengths                                                                                            │
│  2. Resume weaknesses                                                                                           │
│  3. ATS optimization suggestions                                                                                │
│  4. Important keywords for the target role                                                                      │
│  5. Project improvement suggestions                                                                             │
│  6. Experience improvement suggestions                                                                          │
│  7. Formatting suggestions                                                                                      │
│  8. Skills that should be highlighted                                                                           │
│  9. Final improvement checklist                                                                                 │
│                                                                                                                 │
│  Do not invent qualifications, experience, projects, or skills.                                                 │
│                                                                                                                 │
│  Agent: Resume Reviewer                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ Final Resume Analysis:


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1b03a553-9ac2-4a22-968e-bf73f952ee4e                                                                       │
│  Final Output: ### Resume Improvement Report for Vallur Sree Vardhan                                            │
│                                                                                                                 │
│  #### 1. Resume Strengths                                                                                       │
│                                                                                                                 │
│  - **Technical Proficiency**: Vallur possesses a solid foundation in both frontend and backend technologies,    │
│  with strong skills in React.js, Node.js, Express.js, and database management with MongoDB and MySQL.           │
│  - **Project Management**: Demonstrated project management skills through leading the development of            │
│  AI-powered code review tools and a Chrome extension generator.                                                 │
│  - **AI Integration**: Extensive experience with integrating AI-related APIs and practical application in       │
│  real-world projects.                                                                                           │
│  - **Certifications**: Holds relevant certifications that enhance technical and practical skills, such as       │
│  those from Forage and Infosys Springboard.                                                                     │
│  - **Active Participation**: Active participant in beta testing communities, contributing to the development    │
│  of innovative tools.                                                                                           │
│                                                                                                                 │
│  #### 2. Resume Weaknesses                                                                                      │
│                                                                                                                 │
│  - **Lack of Professional Work Experience**: No full-time professional work experience apart from the beta      │
│  testing role, which might be a concern for some employers.                                                     │
│  - **Advanced Topics**: Limited exposure to advanced topics such as machine learning, database optimization,    │
│  and large-scale system architecture.                                                                           │
│  - **DevOps and Cloud Experience**: No mention of experience with CI/CD pipelines, containerization,            │
│  orchestration, or major cloud platforms like AWS, GCP, or Azure.                                               │
│  - **Security Skills**: No specific skills mentioned in advanced security practices or compliance.              │
│                                                                                                                 │
│  #### 3. ATS Optimization Suggestions                                                                           │
│                                                                                                                 │
│  - **Keywords**: Use keywords directly from the job description in the resume. Incorporate terms like "full     │
│  stack developer," "React.js," "Node.js," "Express.js," "RESTful APIs," "JWT Authentication," "MongoDB,"        │
│  "MySQL," "CI/CD," "Docker," "AWS," "security practice


### Resume Improvement Report for Vallur Sree Vardhan

#### 1. Resume Strengths

- **Technical Proficiency**: Vallur possesses a solid foundation in both frontend and backend technologies, with strong skills in React.js, Node.js, Express.js, and database management with MongoDB and MySQL.
- **Project Management**: Demonstrated project management skills through leading the development of AI-powered code review tools and a Chrome extension generator.
- **AI Integration**: Extensive experience with integrating AI-related APIs and practical application in real-world projects.
- **Certifications**: Holds relevant certifications that enhance technical and practical skills, such as those from Forage and Infosys Springboard.
- **Active Participation**: Active participant in beta testing communities, contributing to the development of innovative tools.

#### 2. Resume Weaknesses

- **Lack of Professional Work Experience**: No full-time professional work experience apart from the beta testing r